# Build Point-in-Time Universe

Builds the monthly point-in-time crypto universe used by the backtest, removing the survivorship bias of the original dataset.

**Spec:** [`specs/08_universe.md`](../specs/08_universe.md)

**Ranking signal:** rolling 30-day Binance USDT quote volume (tradable-liquidity proxy). See the spec's methodology note for why we use Binance volume instead of CoinGecko market cap.

## Inputs
- `data/coingecko_candidates.json` — curated candidate list of ~70 coins plausibly in the top-50 during 2018-2024.
- `data/binance_listings_manual.json` — hand-curated Binance USDT listing / delisting dates.
- `data/binance_usdt_pairs_2018-12-31_2024-01-01_1d.csv` — existing daily klines for 50 surviving USDT pairs.
- `data/binance_pit_supplement_2019-2024_1d.csv` — daily klines for delisted / late-listed pairs (built by this notebook).

## Outputs
- `data/pit_universe.csv` — long-format universe with `(date, symbol, rolling_quote_vol_30d, included)`.
- `data/pit_universe_summary.csv` — one row per monthly snapshot.

In [ ]:
import json
import pandas as pd

from src.universe import (
    fetch_binance_extended_prices,
    to_monthly_snapshots,
    build_pit_universe,
    summarize_pit_universe,
    load_binance_listings,
    DEFAULT_EXCLUDED_SYMBOLS,
)

EXISTING_CSV = '../data/binance_usdt_pairs_2018-12-31_2024-01-01_1d.csv'
SUPPLEMENT_CSV = '../data/binance_pit_supplement_2019-2024_1d.csv'
CANDIDATES_JSON = '../data/coingecko_candidates.json'
LISTINGS_JSON = '../data/binance_listings_manual.json'
PIT_OUT = '../data/pit_universe.csv'
SUMMARY_OUT = '../data/pit_universe_summary.csv'

WINDOW_START = '2019-01-01'
WINDOW_END = '2024-01-01'

## 1. Identify symbols needing supplementary fetch

In [ ]:
candidates = json.load(open(CANDIDATES_JSON))['candidates']
candidate_syms = sorted({c['symbol'] for c in candidates})

existing = pd.read_csv(EXISTING_CSV, usecols=['symbol'])
existing_bare = {s.replace('USDT', '') for s in existing['symbol'].unique()}

missing = sorted(set(candidate_syms) - existing_bare)
print(f'candidates: {len(candidate_syms)}')
print(f'already in existing CSV: {len(set(candidate_syms) & existing_bare)}')
print(f'need supplementary fetch: {len(missing)}')
print(missing)

## 2. Fetch supplementary klines (delisted + late-listed pairs)

Cached: re-runs read from `SUPPLEMENT_CSV` and skip symbols already present. Symbols that never traded on Binance USDT spot (LEO, HT, KCS, OKB, BSV, CRO, TON) will fail gracefully and be excluded from the universe — which is the correct PIT behavior since they couldn't have been traded.

In [ ]:
supplement = fetch_binance_extended_prices(
    symbols=missing,
    start=WINDOW_START,
    end=WINDOW_END,
    cache_path=SUPPLEMENT_CSV,
    delay=0.3,
)
print(f'supplementary rows: {len(supplement)}')
print(f'symbols fetched: {sorted(supplement["symbol"].unique())}')

## 3. Combine existing + supplementary klines

In [ ]:
existing_full = pd.read_csv(EXISTING_CSV, parse_dates=['open_time'])
all_prices = pd.concat([existing_full, supplement], ignore_index=True)
all_prices = all_prices.drop_duplicates(subset=['symbol', 'open_time'])
print(f'combined rows: {len(all_prices)}')
print(f'unique symbols: {all_prices["symbol"].nunique()}')

## 4. Compute monthly snapshots (rolling 30d quote volume)

In [ ]:
monthly = to_monthly_snapshots(all_prices)
print(f'monthly rows: {len(monthly)}, snapshots: {monthly["snapshot_date"].nunique()}')
print(f'date range: {monthly["snapshot_date"].min().date()} -> {monthly["snapshot_date"].max().date()}')
monthly.head()

## 5. Build the PIT universe

In [ ]:
listings = load_binance_listings(LISTINGS_JSON)

pit = build_pit_universe(
    monthly,
    top_n=30,
    min_age_days=180,
    min_median_volume_usd=1_000_000,
    excluded_symbols=DEFAULT_EXCLUDED_SYMBOLS,
    entry_buffer_months=2,
    exit_buffer_months=1,
    binance_listings=listings,
)
print(f'PIT rows: {len(pit)}')
print(f'distinct symbols ever included: {pit[pit["included"]]["symbol"].nunique()}')
pit.head(10)

## 6. Persist artifacts

In [ ]:
pit.to_csv(PIT_OUT, index=False)
summary = summarize_pit_universe(pit)
summary.to_csv(SUMMARY_OUT, index=False)
print(f'wrote {PIT_OUT} ({len(pit)} rows)')
print(f'wrote {SUMMARY_OUT} ({len(summary)} rows)')

## 7. Acceptance-criteria checks (Spec 08 §4)

In [ ]:
included = pit[pit['included']]['symbol'].unique()

# AC1: contains at least one significant delisted asset
ac1 = 'LUNAUSDT' in included or 'FTTUSDT' in included
print(f'AC1 (delisted in universe): LUNA={"LUNAUSDT" in included}, FTT={"FTTUSDT" in included} -> {"PASS" if ac1 else "FAIL"}')

# AC2: median size ~30 ± 3 (steady-state, after warmup)
steady = summary['n_included'].iloc[18:]  # skip pre-2020-08 warmup
ac2 = abs(steady.median() - 30) <= 3
print(f'AC2 (steady-state size ~30 ± 3): median={steady.median()}, range=[{steady.min()},{steady.max()}] -> {"PASS" if ac2 else "FAIL"}')

# AC3: no stables
stables = {'USDTUSDT','USDCUSDT','BUSDUSDT','DAIUSDT','TUSDUSDT','USDPUSDT','FDUSDUSDT'}
bad = set(included) & stables
print(f'AC3 (no stables): {"PASS" if not bad else f"FAIL — found {bad}"}')

# AC6: >=10 assets ever-in but absent at end
end_date = pit['date'].max()
ever_in = set(pit[pit['included']]['symbol'])
at_end = set(pit[(pit['date']==end_date) & pit['included']]['symbol'])
gone = sorted(ever_in - at_end)
ac6 = len(gone) >= 10
print(f'AC6 (>=10 ever-in but absent at end): {len(gone)} -> {"PASS" if ac6 else "FAIL"}')
print(f'   gone: {gone}')

## 8. Visual diagnostic — universe membership over time

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
summary.set_index('date')[['n_eligible', 'n_included']].plot(ax=ax, marker='o', markersize=3)
ax.axhline(30, color='grey', linestyle='--', alpha=0.5, label='target N=30')
ax.set_ylabel('number of symbols')
ax.set_title('PIT universe size over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../paper/figures/pit_universe_evolution.png', dpi=130, bbox_inches='tight')
plt.show()